## Language Detection: Preliminary Investigation

This notebook focus on the preliminary investigation for detecting language.

Author: Boon Heng Chew <<s2831745@sms.ed.ac.uk>>

In [10]:
from platform import python_version

# show environment
print(f"Python environment: {python_version()}")

Python environment: 3.12.4


In [11]:
# setup the test data for language detection
test_files = [
    # Programming languages
    ("script.py", "#!/usr/bin/env python3\nimport sys\nprint('Hello World')"),
    ("main.go", "package main\nfunc main() {\n\tfmt.Println(\"Hello\")\n}"),
    ("app.js", "const express = require('express');\napp.listen(3000);"),
    ("Main.java", "public class Main {\n  public static void main(String[] args) {}\n}"),
    ("test.rs", "fn main() {\n    println!(\"Hello\");\n}"),
    ("main.c", "#include <stdio.h>\nint main() { return 0; }"),
    ("main.cpp", "#include <iostream>\nint main() { std::cout << \"Hi\"; }"),
    ("script.sh", "#!/bin/bash\necho 'test'"),
    
    # Markup/Styling
    ("style.css", "body { margin: 0; padding: 0; }"),
    ("styles.scss", "$primary: #333;\nbody { color: $primary; }"),
    ("index.html", "<!DOCTYPE html>\n<html><body>Test</body></html>"),
    ("component.tsx", "export const App: React.FC = () => <div>Hello</div>"),
    
    # Configuration/Data
    ("Makefile", ".PHONY: all\nall:\n\t@echo \"Building\""),
    ("config.yaml", "version: 1.0\nserver:\n  port: 8080"),
    ("data.json", '{"name": "test", "value": 42}'),
    (".gitignore", "*.pyc\n__pycache__/\n.env"),
    ("Dockerfile", "FROM python:3.11\nRUN pip install flask"),
    ("requirements.txt", "flask==2.0.1\nrequests>=2.25.0"),
    ("package.json", '{"name": "app", "version": "1.0.0"}'),
    ("CMakeLists.txt", "cmake_minimum_required(VERSION 3.10)\nproject(MyProject)"),
    
    # Documentation
    ("README.md", "# Project Title\n\nDescription here"),
    ("docs/api.md", "# API Documentation\n\n## Endpoints"),
    ("CHANGELOG.txt", "Version 1.0\n- Initial release"),
    
    # Vendored files (third-party code)
    ("vendor/lib.js", "// Third party library\nfunction external() {}"),
    ("node_modules/pkg/index.js", "module.exports = {};"),
    ("third_party/component.py", "# External dependency\nclass Lib: pass"),
    
    # Generated files
    ("generated_pb.py", "# -*- coding: utf-8 -*-\n# Generated by the protocol buffer compiler.  DO NOT EDIT!"),
    ("output.min.js", "!function(){console.log('minified')}()"),
    ("bundle.js", "// Generated bundle - do not edit\nmodule.exports={}"),
    ]

### Exploring pygments

This section focus on using pygments for language detection. It is a pure python library but it is known to have limitation. For instance, it can't detect Dockerfile and that it can't differentiate groovy from gradle file. 

In [12]:
# library and dependencies
import os
from pygments.lexers import guess_lexer_for_filename, get_lexer_by_name
from pygments.lexers import get_all_lexers, find_lexer_class
from pygments.util import ClassNotFound

In [13]:
def pygment_detect_language(filepath: str, content: str = None):

    try:
        # read source code
        if content is None:
            try:
                with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
                    content = f.read()
            except(IOError, OSError):
                content = ""

        basename = os.path.basename(filepath).lower()
        
        # For very short snippets, Pygments struggles - use extension-based detection
        if content and len(content.strip()) < 50:
            # Try to get lexer by extension only first
            ext = os.path.splitext(basename)[1].lower()
            ext_lexer_map = {
                '.py': 'python', '.js': 'javascript', '.ts': 'typescript',
                '.java': 'java', '.c': 'c', '.cpp': 'cpp', '.cc': 'cpp',
                '.go': 'go', '.rs': 'rust', '.rb': 'ruby', '.php': 'php',
                '.sh': 'bash', '.yaml': 'yaml', '.yml': 'yaml',
                '.json': 'json', '.xml': 'xml', '.html': 'html',
                '.css': 'css', '.sql': 'sql', '.vue': 'vue', '.toml': 'toml',
            }
            if ext in ext_lexer_map:
                try:
                    lexer = get_lexer_by_name(ext_lexer_map[ext])
                    return lexer.name
                except:
                    pass  # Fall through to normal detection
        
        
        lexer = guess_lexer_for_filename(basename, content)
        return lexer.name
        
    except ClassNotFound:
        # Pygments couldn't determine language
        return "Unknown"
    # except Exception as e:
    #     # Any other error (encoding issues, etc.)
    #     return "Unknown"
    return "Unknown"
    

In [14]:
# Testing with pygment directly

print("Testing language detection with pygment:\n")
for filepath, content in test_files:
    lang = pygment_detect_language(filepath, content)
    # lang_conf, conf = detect_language_with_confidence(filepath, content)
    # adapter = get_adapter_name(lang)
    lang_conf =""
    conf = 1
    
    print(f"{filepath:25s} -> {lang:15s} (confidence: {conf:.2f})")


Testing language detection with pygment:

script.py                 -> Python          (confidence: 1.00)
main.go                   -> Go              (confidence: 1.00)
app.js                    -> JavaScript      (confidence: 1.00)
Main.java                 -> Java            (confidence: 1.00)
test.rs                   -> Rust            (confidence: 1.00)
main.c                    -> C               (confidence: 1.00)
main.cpp                  -> C++             (confidence: 1.00)
script.sh                 -> Bash            (confidence: 1.00)
style.css                 -> CSS             (confidence: 1.00)
styles.scss               -> SCSS            (confidence: 1.00)
index.html                -> HTML            (confidence: 1.00)
component.tsx             -> TSX             (confidence: 1.00)
Makefile                  -> Makefile        (confidence: 1.00)
config.yaml               -> YAML            (confidence: 1.00)
data.json                 -> JSON            (confidence: 1.00

### Augmenting pygment with lookup tables

The shortcoming of pygment probably can be filled by augmenting it with a lookup table of commonly known files (eg. `Dockerfile` files are rather fixed, same applied for `pom.xml` `build.gradle`, etc)

In [15]:
COMMON_FILENAME_MAP = {
    'makefile': 'Makefile',
    'dockerfile': 'Dockerfile',
    'cmakelists.txt': 'CMake',
    'requirements.txt': 'pip requirements',
    'gemfile': 'Ruby',
    'rakefile': 'Ruby',
    'package.json': 'JSON',
    'composer.json': 'JSON',
    'cargo.toml': 'TOML',
    '.gitignore': 'gitignore',
    '.gitattributes': 'gitignore',
    '.dockerignore': 'gitignore',
    '.eslintrc': 'JSON',
    '.babelrc': 'JSON',
    'readme.md': 'Markdown',
    'changelog.md': 'Markdown',
    'changelog.txt': 'Text',
    'readme.txt': 'Text',
}

# fallbacks for less common cases 
# (well known filename extension tha pygment does not do well in)
COMMON_FILENAME_EXTENSION_MAP = {
    '.py': 'Python',
    '.pyw': 'Python',
    '.js': 'JavaScript',
    '.jsx': 'JSX',
    '.ts': 'TypeScript',
    '.tsx': 'TypeScript',
    '.java': 'Java',
    '.c': 'C',
    '.h': 'C',
    '.cpp': 'C++',
    '.cc': 'C++',
    '.cxx': 'C++',
    '.hpp': 'C++',
    '.go': 'Go',
    '.rs': 'Rust',
    '.rb': 'Ruby',
    '.php': 'PHP',
    '.sh': 'Bash',
    '.bash': 'Bash',
    '.zsh': 'Bash',
    '.yaml': 'YAML',
    '.yml': 'YAML',
    '.json': 'JSON',
    '.xml': 'XML',
    '.html': 'HTML',
    '.htm': 'HTML',
    '.css': 'CSS',
    '.scss': 'SCSS',
    '.sass': 'Sass',
    '.less': 'LESS',
    '.sql': 'SQL',
    '.vue': 'Vue',
    '.toml': 'TOML',
    '.md': 'Markdown',
    '.markdown': 'Markdown',
    '.rst': 'reStructuredText',
    '.swift': 'Swift',
    '.kt': 'Kotlin',
    '.kts': 'Kotlin',
    '.scala': 'Scala',
    '.r': 'R',
    '.lua': 'Lua',
    '.pl': 'Perl',
    '.m': 'Objective-C',  # or MATLAB - ambiguous
    '.gradle': 'Groovy',
    '.cs': 'C#',
    '.fs': 'F#',
    '.clj': 'Clojure',
    '.ex': 'Elixir',
    '.exs': 'Elixir',
    '.erl': 'Erlang',
    '.hs': 'Haskell',
    '.elm': 'Elm',
    '.dart': 'Dart',
}

# Compound extensions
COMPOUND_EXTENSION_MAP = {
    '.gradle.kts': 'Kotlin',
    '.spec.ts': 'TypeScript',
    '.test.ts': 'TypeScript',
    '.spec.js': 'JavaScript',
    '.test.js': 'JavaScript',
    '.min.js': 'JavaScript',
    '.config.js': 'JavaScript',
}

In [16]:
def pygment_detect_language_enhanced(filepath: str, content: str = None) -> tuple[str, float]:
    """
    Strategy:
    1. Check exact filename match in lookup table
    2. Check file extension in lookup table
    3. Use Pygments guess_lexer_for_filename (filename + content analysis)
    """

    basename = os.path.basename(filepath).lower()

    # based on filename convention, determine it is that sort of file
    if basename in COMMON_FILENAME_MAP:
        return COMMON_FILENAME_MAP[basename], 1.0

    # Check for compound extensions (.gradle.kts)
    for extension_pattern, lang in COMPOUND_EXTENSION_MAP.items():
        if basename.endswith(extension_pattern):
            return lang, 1.0
        
    # 3. Check filename extension / convention
    ext = os.path.splitext(basename)[1].lower()
    if ext in COMMON_FILENAME_EXTENSION_MAP:
        return COMMON_FILENAME_EXTENSION_MAP[ext], 0.95
    
    # detect based on file content for file without extension or unknown extension
    if content is None:
        try:
            with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
                content = f.read()
        except(IOError, OSError):
            content = ""

    try:
        lexer = guess_lexer_for_filename(filepath, content or "", stripnl=False)
    
        # Map Pygments lexer names to cleaner names
        name_map = {
            'Text only': 'Text',
            'GDScript': 'Unknown',  # Common false positive
            'Arduino': 'C++',  # Arduino is C++ dialect
            'CSS+Lasso': 'SCSS',  # Fix common SCSS misdetection
            'scdoc': 'Unknown',  # Common false positive for Python
        }
    
        lang_name = name_map.get(lexer.name, lexer.name)
    
        # Lower confidence for ambiguous detections
        if lang_name in ['Unknown', 'Text']:
            return lang_name, 0.3
    
        return lang_name, 0.7
        
    except ClassNotFound:
        # Pygments couldn't determine language
        return "Unknown", 0.0
    except Exception as e:
        return "Unknown", 0.0

In [17]:
# Testing with pygment directly

print("Testing language detection with pygment:\n")
for filepath, content in test_files:
    lang, conf = pygment_detect_language_enhanced(filepath, content)
    
    print(f"{filepath:25s} -> {lang:15s} (confidence: {conf:.2f})")

Testing language detection with pygment:

script.py                 -> Python          (confidence: 0.95)
main.go                   -> Go              (confidence: 0.95)
app.js                    -> JavaScript      (confidence: 0.95)
Main.java                 -> Java            (confidence: 0.95)
test.rs                   -> Rust            (confidence: 0.95)
main.c                    -> C               (confidence: 0.95)
main.cpp                  -> C++             (confidence: 0.95)
script.sh                 -> Bash            (confidence: 0.95)
style.css                 -> CSS             (confidence: 0.95)
styles.scss               -> SCSS            (confidence: 0.95)
index.html                -> HTML            (confidence: 0.95)
component.tsx             -> TypeScript      (confidence: 0.95)
Makefile                  -> Makefile        (confidence: 1.00)
config.yaml               -> YAML            (confidence: 0.95)
data.json                 -> JSON            (confidence: 0.95

### Exploring guesslang

This section focus on using guesslang for language detection. It ia a ML based approach for language detection, and has been used by VS Code for language detection file load.

In [ ]:
%%bash
docker run --rm --platform linux/amd64 tensorflow/tensorflow:2.5.0 bash -c '
pip install -q guesslang==2.2.1 && \
python3 << EOF
from guesslang import Guess

test_files = [
        ("Dockerfile", None),
        ("main.py", "def hello():\n    print('hello')"),
        ("app.js", "console.log('hello');"),
        ("CMakeLists.txt", "cmake_minimum_required(VERSION 3.10)"),
        ("build.gradle", "plugins { id 'java' }"),
        ("requirements.txt", "flask==2.0.0\nnumpy"),
        ("test.c", "#include <stdio.h>\nint main() {}"),
        ("test.cpp", "#include <iostream>\nint main() {}"),
        ("test.go", "package main\nfunc main() {}"),
    ]

guess = Guess()
print(f"{'File':<20} {'Detected':<15}")
print("=" * 35)
for filename, content in test_files:
    detected = guess.language_name(content)
    print(f"{filename:<20} {detected:<15}")
EOF
'

Unable to find image 'tensorflow/tensorflow:2.5.0' locally
2.5.0: Pulling from tensorflow/tensorflow
01bf7da0a88c: Pulling fs layer
f3b4a5f15c7a: Pulling fs layer
57ffbe87baa1: Pulling fs layer
82b03aaebc12: Pulling fs layer
ded4361a2673: Pulling fs layer
bea159c77425: Pulling fs layer
cec93399b3cf: Pulling fs layer
8c7aca956461: Pulling fs layer
0297cbd0f431: Pulling fs layer
c1f5822d5655: Pulling fs layer
82b03aaebc12: Waiting
ded4361a2673: Waiting
bea159c77425: Waiting
cec93399b3cf: Waiting
8c7aca956461: Waiting
0297cbd0f431: Waiting
c1f5822d5655: Waiting
f3b4a5f15c7a: Verifying Checksum
f3b4a5f15c7a: Download complete
57ffbe87baa1: Download complete
01bf7da0a88c: Download complete
82b03aaebc12: Verifying Checksum
82b03aaebc12: Download complete
01bf7da0a88c: Pull complete
cec93399b3cf: Verifying Checksum
cec93399b3cf: Download complete
f3b4a5f15c7a: Pull complete
57ffbe87baa1: Pull complete
82b03aaebc12: Pull complete
bea159c77425: Verifying Checksum
bea159c77425: Download complete

#### Finding 1: Guesslang is broken and no longer maintained

`Guesslang` has a hard dependency on tensorflow but it did a hardpin to particular tensorflow version that no longer with Python 3.9 or beyond. Guesslang with older version of python can't work as well as tensorflow binding for MacOS M series does not exist before Python 3.9. 

It is tried both using docker (with Rosetta Emulation for it to function as linux/amd) as well as on Google Colab. All failed to run the tool.

### Exploring linguist

This section focus on using linguist (https://github.com/github-linguist/linguist) for language detection. It is developed by GitHub but used for language detection at repo level.

This is the backend techology that powered the github colour bar (showing different language and their distribution in the repo). It uses multiple strategies for detection (It has a massive, predefined YAML configuration file, which is similar to a lookup, and a trained classifier). It also uses heuristics by looking at the filename, their convention and shebang (i.e. first line in the file)

Gitlab build their own version of this by forking from `linguist` in the past before they transit to `Gitaly` / `go-enry`, a Go port of `linguist`.

This tool is explored by calling it via a subprocess in Python. It call the ruby implementation in a docker container that contains the `linguist`'s ruby gem

In [ ]:
import subprocess
import tempfile
import os

def linguist_detect_language(files):
    """
    Use a subprocess to trigger Linguist in docker container for languages detection.
    """

    # The files need to exist and linguist work at repo level
    # thus create the files in tmp directory
    with tempfile.TemporaryDirectory() as tmpdir:
        for filename, content in files:
            
            filepath = os.path.join(tmpdir, filename)
            parent_dir = os.path.dirname(filepath)
            if parent_dir:
                os.makedirs(os.path.dirname(filepath), exist_ok=True)

            with open(filepath, 'w') as f:
                f.write(content)

        # Run linguist with breakdown
        cmd = [
            'docker', 'run', '--rm', 
            '--entrypoint', 'sh',
            '-v', f'{tmpdir}:/repo',
            '-w', '/repo',
            'crazymax/linguist',
            '-c',
            'git config --global user.email "t@t.com" && '
            'git config --global user.name "T" && '
            'git config --global init.defaultBranch main && '
            'git init -q && '
            'git add -A && '
            'git commit -qm "init" && '
            'github-linguist --breakdown'
        ]
        
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
        
        if result.returncode != 0:
            print(f"Error running Linguist:")
            print(f"STDERR: {result.stderr}")
            return {}
        
        # Debug: show raw output
        print("=== Raw Linguist Output ===")
        print(result.stdout)
        print("=== End Output ===\n")
        
        # Parse the breakdown format
        file_langs = {}
        current_lang = None
        
        # Parse individual file mappings
        file_langs = {}
        in_file_section = False
        current_lang = None
        
        for line in result.stdout.split('\n'):
            line = line.strip()
            if not line:
                continue
            
            if line.endswith(':') and '/' not in line and not line.startswith('['):
                current_lang = line[:-1]
                in_file_section = True
            elif in_file_section and current_lang:
                file_langs[line] = current_lang
        
        return file_langs

# Run it
results = linguist_detect_language(test_files)

import json
print(json.dumps(results, indent=2))

#### Finding 2: Linguist has high accuracy but it has its shortcoming

Using linguist directly is supposed to be effective for detecting programming language. It is also able to differentiate vendored files, source vs documentation, and even whether code is generated. However, it has the following shortcoming:
- Heavyweight
- Ruby based, and thus, need to do cross language function call
- Function at repo level and requires the directory to be a git repo
    - create problem if using subprocess as there is a need to parse standard output
    - not all files are outputted as the default run show top lang. statistics
- Slow for large scale code analysis (ie. Ruby and use of subprocess)

### Enry (go port of linguist)

This section focus on using Enry (https://github.com/go-enry/go-enry) for language detection. It is 2X faster in performance, and uses the same algorithims and data as `linguist`. It is implemented by regenerating `linguist` internal data structure from its YAML failes (which "function" as a lookup table). It employs multiple strategies from looking file extensions, filenames, shebangs, modelines, content heuristics and a Bayesian classification model. 

In short, go-enry is production grade and best for performance critical application. It should have near identical accuracy to `linguist` but run much faster. FFI bindings exist for multiple languages including Python.
 
In the investigation, it is found that the python binding is no longer maintained (version 0.1.1 is last released in 2020, appears abandoned) but it is possible to use native binding via c-shared library directly by building it locally, bypassing the Python wrapper entirely.This is the approach taken in the experimentation.

### Setup

The experimentation uses Go binding for a native experience of using Entry in Python. Run the following steps for generate the c-shared library before using FFI in python.

```sh
# clone the go-enry repository
git clone https://github.com/go-enry/go-enry.git

# go the shared directory inside go-enry
cd go-enry/shared

# Enable CGO and uses arm64 to build native MacOS M series version (local machine) 
# of the c-shared library for using enry (CGO is the bridge between C and Go)
CGO_ENABLED=1 GOARCH=arm64 go build -buildmode=c-shared -o ../libenry.dylib enry.go

# move the built shared library and header file to local/current directory of this notebook
mv ../libenry.dylib ../../libenry.dylib
mv ../libenry.h ../../libenry.h
```

In [ ]:
import os
from cffi import FFI

ffi = FFI()

# Define the C-interface exactly as Go exported it
ffi.cdef("""
    typedef struct { const char *p; long long n; } GoString;
    typedef struct { void *data; long long len; long long cap; } GoSlice;
    
    extern GoString GetLanguage(GoString filename, GoSlice content);
    extern GoString GetLanguageByExtension(GoString filename);
    extern GoString GetLanguageByFilename(GoString filename);
    extern int IsBinary(GoSlice content);
    extern int IsVendor(GoString filename);
    extern int IsGenerated(GoString filename, GoSlice content);
    extern int IsDocumentation(GoString filename);
    extern GoString GetLanguageType(GoString language);
""")

class Enry:
    def __init__(self, lib_path="./libenry.dylib"):
        abs_path = os.path.abspath(lib_path)
        if not os.path.exists(abs_path):
            raise FileNotFoundError(f"libenry not found at {abs_path}")
        self.lib = ffi.dlopen(abs_path)

    def _make_gostring(self, s: str) -> tuple:
        """Create GoString struct from Python string"""
        b = s.encode('utf-8')
        gs = ffi.new("GoString*")
        gs.p = ffi.from_buffer(b)
        gs.n = len(b)
        return gs, b
    
    def _make_goslice(self, s: str) -> tuple:
        """Create GoSlice struct from Python string"""
        b = s.encode('utf-8')
        sl = ffi.new("GoSlice*")
        sl.data = ffi.from_buffer(b)
        sl.len = len(b)
        sl.cap = len(b)
        return sl, b
    
    def _gostring_to_str(self, gs) -> str:
        """Convert GoString result to Python string"""
        if gs.n == 0:
            return ""
        return ffi.string(gs.p, gs.n).decode('utf-8')

    def detect(self, filename: str, content: str = "") -> str:
        """Language detection using all strategies"""
        
        gs_name, b_name = self._make_gostring(filename)
        sl_cont, b_cont = self._make_goslice(content)

        
        # invoke the go rountine (note: it returns a GoString struct, which CFFI handles)
        result = self.lib.GetLanguage(gs_name[0], sl_cont[0])
        
        # Convert the returned GoString back to Python str
        return self._gostring_to_str(result)
    
    def detect_by_extension(self, filename: str) -> str:
        """Detect language only by file extension"""

        gs, b = self._make_gostring(filename)
        result = self.lib.GetLanguageByExtension(gs[0])
        
        return self._gostring_to_str(result)
    
    def detect_by_filename(self, filename: str) -> str:
        """Detect language by full filename (e.g., Makefile)"""
        
        gs, b = self._make_gostring(filename)
        result = self.lib.GetLanguageByFilename(gs[0])
        
        return self._gostring_to_str(result)
    
    def is_binary(self, content: str) -> bool:
        """Check if content is binary"""
        
        sl, b = self._make_goslice(content)
        return bool(self.lib.IsBinary(sl[0]))
    
    def is_vendored(self, filename: str) -> bool:
        """Check if file is vendored (third-party code)"""

        gs, b = self._make_gostring(filename)
        return bool(self.lib.IsVendor(gs[0]))
    
    def is_generated(self, filename: str, content: str = "") -> bool:
        """Check if file is auto-generated"""
        
        gs, b_name = self._make_gostring(filename)
        sl, b_cont = self._make_goslice(content)

        return bool(self.lib.IsGenerated(gs[0], sl[0]))
    
    def is_documentation(self, filename: str) -> bool:
        """Check if file is documentation"""
        
        gs, b = self._make_gostring(filename)
        return bool(self.lib.IsDocumentation(gs[0]))
    
    def get_language_type(self, language: str) -> str:
        """Get language type (programming, markup, prose, data)"""

        gs, b = self._make_gostring(language)
        result = self.lib.GetLanguageType(gs[0])
        return self._gostring_to_str(result)

# Usage
enry = Enry()
# for filepath, content in test_files:
#     lang = enry.detect(filepath, content)
#     print(f"{filepath:<20} {lang:<18}")

print(f"\n{'Filename':<30} {'Language':<18} {'Type':<12} {'Flags':<20}")
print("-" * 100)
    
# Process each file
for filepath, content in test_files:
    # Detect language
    lang = enry.detect(filepath, content)
    if not lang:
        lang = "(unknown)"
    
    # Get language type (programming, markup, data, prose)
    lang_type = enry.get_language_type(lang) if lang != "(unknown)" else "-"
    
    # Check file characteristics
    flags = []
    if enry.is_vendored(filepath):
        flags.append("VENDOR")
    if enry.is_generated(filepath, content):
        flags.append("GEN")
    if enry.is_documentation(filepath):
        flags.append("DOC")
    if enry.is_binary(content):
        flags.append("BIN")
    
    flags_str = ",".join(flags) if flags else "-"
    
    print(f"{filepath:<30} {lang:<18} {lang_type:<12} {flags_str:<20}")

print("\n" + "=" * 100)
print("Legend:")
print("  Type: programming, markup, prose, data")
print("  Flags: VENDOR (third-party), GEN (auto-generated), DOC (documentation), BIN (binary)")
print("=" * 100)

#### Finding 3: `go-enry` strength and shortcoming

**Strength**:
- go-enry is very fast. It could go down to ~us for file level detection with native binding
- single shared library dependency
- very accuracy and feature rich
- production grade
- short piece of code (essentially a simple rewriting of the FFI wrapper)

**Shortcoming**:
- require building of the c-shared library (essentially, self-maintenance)